# Week 28, Day 6: Forge: Week 28 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w028/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("1.", f"{86e9:.1e} spikes per second")
print("2. gamma =", round(float(np.exp(-1 / (30 * 0.5))), 4))
print("3. N / sqrt(N) =", 400 / np.sqrt(400))

### Exercise: Part A: what each method keeps

Write `cv_acc(X)`: for each trial, compute the mean response to each stimulus over all other trials, assign the trial to the nearest mean (squared distance), and return the fraction correct. Build the calcium-like and voxel measurements (local rng seed 2 for their noise) and store the three accuracies in `acc`.

In [ ]:
r = np.random.default_rng(1)
N, S, trials = 50, 4, 40
pref = r.uniform(0, 2 * np.pi, N); stim_ang = np.arange(S) * np.pi / 2
rate = lambda s: 2 + 18 * np.exp(np.cos(stim_ang[s] - pref) - 1)     # spikes per second
labels = np.repeat(np.arange(S), trials)
counts = np.array([r.poisson(rate(s)) for s in labels]).astype(float)

def cv_acc(X):
    correct = []
    for k in range(len(labels)):
        train = np.arange(len(labels)) != k
        centres = np.array([X[train & (labels == s)].mean(0) for s in range(S)])
        correct.append(...)
    return np.mean(correct)

r2 = np.random.default_rng(2)
calcium = 0.5 * counts + r2.normal(0, 8, counts.shape)
voxel = ...                                       # summed activity plus noise, as a 1-column array
acc = {"spikes": cv_acc(counts), "calcium": cv_acc(calcium), "voxel": cv_acc(voxel)}
print({k: round(float(v), 3) for k, v in acc.items()}, " (chance 0.25)")

In [ ]:
import numpy as np
def test_cv():
    "cv_acc gives leave-one-out nearest-centroid accuracy"
    assert 0 <= acc["spikes"] <= 1 and abs(cv_acc(counts) - acc["spikes"]) < 1e-12

def test_order():
    "spikes decode best, calcium next, and the voxel worst but above the chance level of 0.25"
    assert acc["spikes"] >= acc["calcium"] > acc["voxel"] > 0.3

test_cv()
test_order()
print("All checks passed")

### Exercise: Part B: who gets recorded

Draw 100,000 rates from a log-normal distribution with median 1 Hz and log-sd 1.2 (local rng seed 3). A neuron is recorded if a Poisson count over 60 s reaches at least 50 spikes. Compute the true mean rate `mean_true`, the mean rate of recorded neurons `mean_recorded`, and the fraction never recorded `frac_missed`.

In [ ]:
r3 = np.random.default_rng(3)
rates = np.exp(r3.normal(0.0, 1.2, 100000))
seen = ...
mean_true, mean_recorded = rates.mean(), ...
frac_missed = ...
print(f"true mean {mean_true:.2f} Hz, recorded mean {mean_recorded:.2f} Hz;  {100 * frac_missed:.0f}% of neurons never recorded")
plt.hist(np.log10(rates), 80, alpha=0.5, label="all"); plt.hist(np.log10(rates[seen]), 80, alpha=0.5, label="recorded")
plt.xlabel("log10 rate (Hz)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_bias():
    "the recorded neurons' mean rate is over 1.4 times the population's, and over a third of neurons are never recorded"
    assert mean_recorded > 1.4 * mean_true and frac_missed > 1 / 3

test_bias()
print("All checks passed")